<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/PINN_DeepXDE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install deepxde

## 1.1. Piezoconductivity Equation (Dirichlet)

Solve the 1D piezoconductivity equation $p_t = \kappa p_{xx}$ for $x \in [0, \pi]$, $t \in [0, 1]$, with initial condition $p(x, 0) = \sin(x)$ and boundary conditions $p(0,t) = 0$, $p(\pi,t) = 0$.

In [2]:
import deepxde as dde
import numpy as np
from deepxde.backend import tf

geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

kappa = 1.0
def pde(x, p):
    p_t = dde.grad.jacobian(p, x, i=0, j=1)
    p_xx = dde.grad.hessian(p, x, i=0, j=0)
    return p_t - kappa * p_xx

net = dde.nn.FNN([2] + [20] * 2 + [1], "tanh", "Glorot uniform")

def output_transform(x, y):
    x_col = x[:, 0:1]
    t_col = x[:, 1:2]
    return tf.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

net.apply_output_transform(output_transform)

data = dde.data.TimePDE(
    geomtime, pde, [],
    num_domain=100,
    num_test=None
)

model = dde.Model(data, net)
model.compile("adam", lr=1e-3)

resampler = dde.callbacks.PDEPointResampler(period=1)
losshistory, train_state = model.train(iterations=1000, callbacks=[resampler], display_every=50)

Using backend: tensorflow
Other supported backends: tensorflow.compat.v1, pytorch, jax, paddle.
paddle supports more examples now and is recommended.


Compiling model...
'compile' took 0.039392 s

Training model...

Step      Train loss    Test loss     Test metric
0         [5.45e+00]    [5.45e+00]    []  
50        [3.34e-02]    [3.34e-02]    []  
100       [1.42e-02]    [1.42e-02]    []  
150       [6.06e-03]    [6.06e-03]    []  
200       [2.74e-03]    [2.74e-03]    []  
250       [1.62e-03]    [1.62e-03]    []  
300       [1.25e-03]    [1.25e-03]    []  
350       [1.10e-03]    [1.10e-03]    []  
400       [1.02e-03]    [1.02e-03]    []  
450       [9.66e-04]    [9.66e-04]    []  
500       [9.21e-04]    [9.21e-04]    []  
550       [8.82e-04]    [8.82e-04]    []  
600       [8.46e-04]    [8.46e-04]    []  
650       [8.12e-04]    [8.12e-04]    []  
700       [7.79e-04]    [7.79e-04]    []  
750       [7.47e-04]    [7.47e-04]    []  
800       [7.16e-04]    [7.16e-04]    []  
850       [6.86e-04]    [6.86e-04]    []  
900       [6.57e-04]    [6.57e-04]    []  
950       [6.28e-04]    [6.28e-04]    []  
1000      [5.99e-04]    [

In [3]:
x = np.linspace(0, np.pi, 200)
t = np.linspace(0, 1, 100)
X, T = np.meshgrid(x, t)
inputs = np.stack([X.ravel(), T.ravel()], axis=1)

p_pred = model.predict(inputs).reshape(X.shape)
p_exact = np.exp(-T) * np.sin(X)
err = np.linalg.norm(p_pred - p_exact) / np.linalg.norm(p_exact)
print(f"Relative L2 error = {err:.4e}")

Relative L2 error = 6.5019e-03


In [4]:
import plotly.express as px
import pandas as pd
import numpy as np

domain_x = [0, np.pi]
domain_t = [0, 1]

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
    x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
    t_plot = t_val * np.ones_like(x_plot)
    inputs_plot = np.concatenate([x_plot, t_plot], axis=1)

    p_plot = model.predict(inputs_plot)

    p_exact = np.exp(-t_val) * np.sin(x_plot)
    df_temp = pd.DataFrame({
        'x': x_plot.flatten(),
        'PINN': p_plot.flatten(),
        'Exact': p_exact.flatten(),
        'Time': t_val
    })
    plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')

fig = px.line(df_melted, x='x', y='p', animation_frame='Time',
              color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(xaxis_title='x', yaxis_title='p(x, t)')
fig.show()

## Loss:

In [5]:
import plotly.graph_objects as go
import numpy as np

losses = np.array(losshistory.loss_train).flatten()
steps = np.array(losshistory.steps)

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=losses, mode='lines+markers', name='Dirichlet'))
fig.update_layout(
    title='Сходимость обучения PINN (DeepXDE)',
    xaxis_title='Итерация',
    yaxis_title='Функция потерь',
    yaxis_type='log',
    template='plotly_white'
)
fig.show()

## Что будет если менять число эпох?

In [7]:
import numpy as np
import deepxde as dde

resampler = dde.callbacks.PDEPointResampler(period=1)
checkpoints = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
errors = {}

for step in checkpoints:
    model.train(iterations=1000, callbacks=[resampler], display_every=1000)
    x = np.linspace(0, np.pi, 200); t = np.linspace(0, 1, 100)
    X, T = np.meshgrid(x, t)
    p_pred = model.predict(np.stack([X.ravel(), T.ravel()], axis=1)).reshape(X.shape)
    p_exact = np.exp(-T) * np.sin(X)
    errors[step] = np.linalg.norm(p_pred - p_exact) / np.linalg.norm(p_exact)
    print(f"Итерация {step} → L2 = {errors[step]:.4e}")

AttributeError: type object 'TrainingDisplay' has no attribute 'train'